# Lab 3.1: Getting Started with Strands Agents

This notebook introduces the core concepts of [Strands Agents](https://strandsagents.com) that you will use throughout the workshop:

1. Creating an agent
2. Choosing a model provider
3. Building tools with `@tool`
4. Using lifecycle hooks
5. Multi-agent swarms

**If you are already familiar with Strands Agents, skim to the closing section**, which assembles the `hotel_agent` that Labs 4 and 5 extend.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

import os

import boto3
from dotenv import load_dotenv

load_dotenv()

# Every agent turn below calls Amazon Bedrock, and the two sections that use the
# real retriever also read the graph Lab 1 built. Both are checked once here, so
# an unconfigured environment skips those cells instead of raising in each one.
AGENT_READY = boto3.Session().get_credentials() is not None
NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
GRAPH_READY = AGENT_READY and all(os.environ.get(name) for name in NEO4J_VARS)

print("✅ Environment ready" if AGENT_READY else "No AWS credentials found.")
if AGENT_READY and not GRAPH_READY:
    print("Neo4j is not configured, so the grounded-retrieval cells will skip.")

---
## 1. Creating an Agent

A Strands agent combines a large language model (LLM) with tools. The agent uses the LLM to reason about the user's request and decide which tools to call.

By default, Strands uses **Amazon Bedrock** as the model provider. No extra configuration is needed once your AWS credentials are set.

In [ ]:
from strands import Agent

# Create an agent with a system prompt.
# The system prompt defines the agent's personality and instructions.
agent = Agent(
    system_prompt="You are a helpful travel assistant. Answer questions about hotels and travel.",
)

# Talk to the agent. It uses Amazon Bedrock Claude by default.
if AGENT_READY:
    response = agent("What should I consider when booking a hotel in Lisbon?")
    print(response)
else:
    print("Skipping: no AWS credentials.")

---
## 2. Model Providers

Strands supports multiple model providers. This workshop uses Amazon Bedrock, but you can switch to others.

See all providers: [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/)

In [ ]:
from strands import Agent

# Option 1: Amazon Bedrock, the default. No import needed.
agent_bedrock = Agent(system_prompt="You are a helpful assistant.")

# Option 2: Specify a Bedrock model explicitly
agent_specific = Agent(
    model="us.anthropic.claude-sonnet-5",
    system_prompt="You are a helpful assistant.",
)

# Test it
if AGENT_READY:
    response = agent_specific("Say hello in one sentence.")
    print(response)
else:
    print("Skipping: no AWS credentials.")

---
## 3. Creating Tools

Tools are Python functions decorated with `@tool`. The agent reads the function name and docstring to decide when to call each tool.

**Docstrings are critical.** The agent uses them to match user queries to tools. A vague docstring leads to wrong tool selection.

Two of the three tools below return invented strings, which is all a primer needs to show selection working. The third is the real hybrid retriever from Lab 2, imported rather than rewritten. Watching the agent choose between them is the whole point of the section.

See: [Strands Tools Documentation](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/)

In [ ]:
import json

from strands import Agent, tool

from workshop.hybrid_retrieval import search_hotel_knowledge

@tool
def search_hotels(city: str, max_price: int = 500) -> str:
    """Search for available hotels in a city under a maximum price per night."""
    # In a real application, this would query a database.
    # For this demo, we return a simulated response.
    return f"Found 3 hotels in {city} under ${max_price}/night:\n" \
           f"  1. AnyCompany {city} Resort ($95/night)\n" \
           f"  2. AnyCompany {city} Central ($110/night)\n" \
           f"  3. AnyCompany {city} Budget ($65/night)"

@tool
def book_hotel(hotel_name: str, guest_name: str, nights: int = 1) -> str:
    """Book a hotel room for a guest. Returns a booking confirmation ID."""
    total = nights * 95  # Simulated price
    return f"Booking confirmed: {guest_name} at {hotel_name}\n" \
           f"  {nights} night(s), total: ${total}\n" \
           f"  Confirmation ID: BK-001"

@tool
def search_hotel_knowledge_tool(query: str) -> str:
    """Look up amenities, ratings, and policies for a specific named hotel."""
    # The one tool here that is not simulated. It runs Lab 2's hybrid retrieval
    # against the graph and returns bounded JSON facts, not prose.
    return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

# Create an agent with tools
tools = [search_hotels, book_hotel, search_hotel_knowledge_tool]
agent = Agent(
    tools=tools,
    system_prompt="You are a hotel booking assistant.",
)

print("Agent created with 3 tools:", [t.__name__ for t in tools])

In [ ]:
# The agent decides which tool to call based on the query
print("=== Test 1: Hotel search ===")
if AGENT_READY:
    agent("Find me hotels in Lisbon under $100")
else:
    print("Skipping: no AWS credentials.")

# You should see the agent call search_hotels and return the results

In [ ]:
print("=== Test 2: Grounded hotel knowledge (different tool) ===")
if GRAPH_READY:
    agent("What amenities does AnyCompany Cairo Nile View have?")
else:
    print("Skipping: needs AWS credentials and the graph Lab 1 built.")

# The agent should call search_hotel_knowledge_tool, NOT search_hotels.
# Compare the answer with Test 1: these amenities came out of the graph.

In [ ]:
print("=== Test 3: Booking ===")
if AGENT_READY:
    agent("Book AnyCompany Lisbon Resort for Alice for 3 nights")
else:
    print("Skipping: no AWS credentials.")

# The agent should call book_hotel with the correct parameters

---
## 3.5. Token Counting with Strands

Strands Agents provides **native token counting** through `AgentResult.metrics`. No custom helper functions needed.

This is useful for:
- Comparing the cost of a broad prompt against a targeted one
- Measuring token usage of multi-agent workflows
- Tracking prompt caching effectiveness (when enabled)

**Available metrics:**
- `inputTokens`: tokens sent to the model
- `outputTokens`: tokens generated by the model
- `totalTokens`: input plus output
- `cacheReadInputTokens`: tokens read from cache, when prompt caching is on
- `cacheWriteInputTokens`: tokens written to cache, when prompt caching is on

See: [Strands Metrics Documentation](https://strandsagents.com/docs/user-guide/concepts/agents/metrics/)

In [ ]:
from strands import Agent

# Create a simple agent
agent_metrics = Agent(system_prompt="You are a helpful assistant. Keep answers brief.")

if not AGENT_READY:
    print("Skipping: no AWS credentials.")
else:
    # Make a request and capture the result
    result = agent_metrics("What is the capital of Portugal?")

    # Access token metrics
    if result.metrics:
        usage = result.metrics.accumulated_usage
        print(f"💰 Token Usage:")
        print(f"   Input:  {usage['inputTokens']} tokens")
        print(f"   Output: {usage['outputTokens']} tokens")
        print(f"   Total:  {usage['totalTokens']} tokens")

        # Calculate approximate cost (example: Claude Sonnet pricing)
        input_cost = (usage['inputTokens'] / 1_000_000) * 3.00  # $3/MTok
        output_cost = (usage['outputTokens'] / 1_000_000) * 15.00  # $15/MTok
        total_cost = input_cost + output_cost
        print(f"\n   Estimated cost: ${total_cost:.6f}")
    else:
        print("Metrics not available")

    print(f"\n📝 Response: {result}")

---
## 4. Lifecycle Hooks

Hooks intercept the agent's execution at specific points. The most important hook for this workshop is `BeforeToolCallEvent`. It fires after the LLM decides to call a tool and **before** the tool executes.

Setting `event.cancel_tool` prevents the tool from executing. The LLM receives the cancellation message instead of the tool result. **The LLM cannot bypass this.** It happens at the framework level.

Note where the number lives: the limit of 10 is a Python literal inside `MaxGuestsHook` below, sitting in the same file as the agent that enforces it. Lab 4 comes back to that.

See: [Strands Hooks Documentation](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/)

In [ ]:
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent

@tool
def book_room(hotel: str, guests: int = 1) -> str:
    """Book a hotel room for a number of guests."""
    return f"SUCCESS: Booked {hotel} for {guests} guests"

class MaxGuestsHook(HookProvider):
    """Block bookings with more than 10 guests."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check)

    def check(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] == "book_room":
            guests = event.tool_use["input"].get("guests", 1)
            if guests > 10:
                # This BLOCKS the tool call. The LLM cannot override it.
                event.cancel_tool = f"BLOCKED: {guests} guests exceeds maximum of 10"

# Agent WITHOUT hook: no protection
agent_no_hook = Agent(tools=[book_room], system_prompt="You are a booking assistant.")

# Agent WITH hook: enforces the rule
agent_with_hook = Agent(tools=[book_room], hooks=[MaxGuestsHook()], system_prompt="You are a booking assistant.")

print("Created two agents: one without hook, one with MaxGuestsHook")

In [ ]:
print("=== Without hook: books 15 guests (no protection) ===")
if AGENT_READY:
    agent_no_hook("Book AnyCompany Lisbon Resort for 15 guests")
else:
    print("Skipping: no AWS credentials.")

# The agent books 15 guests: no validation

In [ ]:
print("=== With hook: BLOCKS 15 guests ===")
if AGENT_READY:
    agent_with_hook("Book AnyCompany Lisbon Resort for 15 guests")
else:
    print("Skipping: no AWS credentials.")

# The hook blocks the call. The agent reports the error to the user.

---
## 5. Multi-Agent Swarms

A swarm is a group of agents that hand off work to each other. Each agent has a specific role. Strands manages the handoff chain automatically.

See: [Strands Multi-Agent Documentation](https://strandsagents.com/docs/user-guide/concepts/multi-agent/)

In [ ]:
from strands import Agent, tool
from strands.multiagent import Swarm

@tool
def lookup_hotel(name: str) -> str:
    """Look up hotel details by name."""
    hotels = {"AnyCompany Lisbon Resort": "4 stars, $95/night, pool, spa"}
    return hotels.get(name, f"Hotel '{name}' not found in database")

# Three agents with different roles
executor = Agent(
    name="Executor",
    tools=[lookup_hotel],
    system_prompt="You look up hotel information using your tools. Report exactly what the tool returns.",
)

validator = Agent(
    name="Validator",
    system_prompt="You verify if the Executor's response is consistent. Check for fabricated data. Hand off to Critic.",
)

critic = Agent(
    name="Critic",
    system_prompt="You give a final verdict: VALID or SUSPICIOUS. Be brief.",
)

swarm = Swarm(nodes=[executor, validator, critic], max_handoffs=5)

print("Swarm created: Executor → Validator → Critic")

In [ ]:
print("=== Valid query ===")
if AGENT_READY:
    swarm("What are the details for AnyCompany Lisbon Resort?")
else:
    print("Skipping: no AWS credentials.")

# Executor looks up the hotel, Validator checks, Critic approves

In [ ]:
print("=== Invalid query (hotel does not exist) ===")
if AGENT_READY:
    swarm("What are the details for AnyCompany Antarctica Lodge?")
else:
    print("Skipping: no AWS credentials.")

# Executor gets 'not found', Validator flags it, Critic marks SUSPICIOUS

---
## Summary

| Concept | What it does | Where the workshop uses it |
|---------|-------------|----------------|
| `Agent` + `@tool` | LLM reasons and calls functions | Labs 3, 4, 5, 6 |
| Model providers | Choose Bedrock, Anthropic, OpenAI, etc. | Every lab that calls a model |
| `BeforeToolCallEvent` + `cancel_tool` | Block tool calls that violate rules | Lab 4 |
| `Swarm` | Multi-agent handoff chain | This notebook only |
| `AgentResult.metrics` | Count tokens and estimate cost | Any lab, when cost matters |

The closing section below assembles `hotel_agent`, the agent Labs 4 and 5 carry forward.

---
## 6. Assembling `hotel_agent`

The five sections above each built their own agent to make one point. This
section builds the one agent the rest of the workshop uses, and it takes exactly
two pieces from what you just saw: the real retriever tool from section 3, and
the guest-limit hook from section 4.

Nothing else carries forward. `search_hotels` and `book_hotel` invented their
answers, which was fine for showing tool selection and is not fine once an agent
is answering about real hotels. `Swarm` stays in this notebook.

Then ask it two questions, one it can answer from the graph and one the hook has
to stop.

In [ ]:
if not AGENT_READY:
    print("Skipping: no AWS credentials.")
else:
    from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS

    hotel_agent = Agent(
        name="hotel_agent",
        model="us.anthropic.claude-sonnet-5",
        tools=[search_hotel_knowledge_tool, book_room],
        hooks=[MaxGuestsHook()],
        system_prompt=(
            "You are a hotel assistant. Call search_hotel_knowledge_tool before "
            "answering any question about a hotel, and take bookings only "
            "through book_room.\n\n" + GROUNDING_INSTRUCTIONS
        ),
    )
    print("hotel_agent created with 2 tools and MaxGuestsHook attached")

In [ ]:
print("=== A question the graph can answer ===")
if GRAPH_READY:
    hotel_agent(
        "What amenities and guest rating does AnyCompany Cairo Nile View have?"
    )
else:
    print("Skipping: needs AWS credentials and the graph Lab 1 built.")

In [ ]:
print("=== A booking the hook has to stop ===")
if AGENT_READY:
    hotel_agent("Book AnyCompany Cairo Nile View for 15 guests")
else:
    print("Skipping: no AWS credentials.")

# The hook cancels the tool call before book_room runs. Lab 4 replaces that
# Python literal with a rule read from the graph.